### RAG pipelines - Data ingestion to DB pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\subha\AppData\Local\Temp\ipykernel_16668\2043801575.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,TextLoader


In [6]:
### Read all pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """process all PDF files in a directory"""
    all_documents = []
    pdf_dir=Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
                
            all_documents.extend(documents)
            print(f"Loaded {(len(documents))} documents from {pdf_file.name}")
            
            
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
            
    return all_documents
        
        
processed_documents = process_all_pdfs("../data")
        

Found 3 PDF files in ../data
Processing ..\data\pdf\attention.pdf...
Loaded 15 documents from attention.pdf
Processing ..\data\pdf\BERT.pdf...
Loaded 16 documents from BERT.pdf
Processing ..\data\pdf\context_RAG.pdf...
Loaded 19 documents from context_RAG.pdf


In [7]:
processed_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [8]:
## Text splitting get converted into chunks

In [9]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks"
)
    return split_docs

In [10]:
chunked_documents = split_documents(processed_documents, chunk_size=1000, chunk_overlap=200)

Split 50 documents into 227 chunks


In [11]:
chunked_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

### embedding and vectorStoreDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer and storage in ChromaDB"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e
        
### initialize the embedding manager

embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
embedding_manager
        
        
        
    
    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4196.95it/s]


Loaded model: all-MiniLM-L6-v2


In [14]:
import os

In [15]:
### vectorStore
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store with a collection name and persistence directory
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # create persistent ChromaDb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            ## Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"}
                )
            print(f"Initialized ChromaDB collection: {self.collection_name}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the collection"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")
        
        print(f"Adding {len(documents)} documents to the collection...")
        
        #prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_texts=[]
        embeddings_list=[]
        
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            
            documents_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the collection.")
            print(f"Collection now contains {self.collection.count()} documents.")
        except Exception as e:
            print(f"Error adding documents to the collection: {e}")
            raise e
        
vectorstore=VectorStore()
vectorstore


Initialized ChromaDB collection: pdf_documents


In [16]:
## convert the text into embeddings

texts = [doc.page_content for doc in chunked_documents]

## generate embeddings for the chunked documents
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector db
vectorstore.add_documents(chunked_documents, embeddings)

Generating embeddings for 227 texts...


Batches: 100%|██████████| 8/8 [00:08<00:00,  1.07s/it]


Adding 227 documents to the collection...
Successfully added 227 documents to the collection.
Collection now contains 454 documents.


### Retriever pipeline from vectorstore

In [17]:
class RAGRetriever:
    """Handles query based retrieval from the vector store"""
    
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager ## for generating query embeddings

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents based on the query"""
        print(f"Retrieving top {top_k} documents for query: '{query}'")
        
        #generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["metadatas", "documents", "distances"]
            )
            
            #process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids= results['ids'][0]
                
                for i,(doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": doc,
                            "metadata": metadata,
                            "score": similarity_score
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}.")
            else:
                print("No documents retrieved from the vector store.")
            return retrieved_docs       
        except Exception as e:
            print(f"Error occurred while retrieving documents: {e}")
            return []   

rag_retriever = RAGRetriever(vectorstore=vectorstore, embedding_manager=embedding_manager)

In [18]:
rag_retriever.retrieve("What is attention is all you need?", top_k=5)

Retrieving top 5 documents for query: 'What is attention is all you need?'
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.52it/s]

Retrieved 4 documents above the score threshold of 0.0.


[{'id': 'doc_0f0a11a7_12',
  'document': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'trapped': '/False',
   'total_pages': 15,
   'file_type': 'pdf',
   'keywords': '',
   'producer': 'pdfTeX-1.40.25',
   'page_label': '3',
   'source': '..\\data\\pdf\\attention.pdf',
   'title': '',
   'page': 2,
   'creationdate': '2024-04-10T21:11:43+00:00',
   'doc_index': 12,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'content_length': 216,
   'source_file': 'attention.pdf',
   'creator': 'LaTeX with hyperref',
   'moddate': '2024-04-10T21:11:43+00:00',
   'subject': '',
   'author': ''},
  'score': 0.12962281703948975},
 {'id': 'doc_8c9c6aca_12',
  'document': '3.2 Attention\nAn attention function can be described as mapping a query a

In [19]:
## now context + prompt is called augmentation as the context is used to augment the prompt for the LLM to generate a response
## as we know A in RAG stands for Augmented, so the context is used to augment the prompt for the LLM to generate a response

### Integration Vectordb Context pipeline with LLM output

In [30]:
## simple rag pipeline with GroqLLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.environ.get("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['document'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found."
    
    prompt=f"""
            You are a helpful assistant. Use the following context to answer the question.
            Context: {context}
            Question: {query}
            Answer:
            """
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [31]:
answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
print("Answer:", answer)

Retrieving top 3 documents for query: 'What is attention mechanism?'
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.53it/s]

Retrieved 3 documents above the score threshold of 0.0.


Answer: The attention mechanism is a function that maps a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weights are determined by the similarity between the query and the keys. This allows the model to focus on the most relevant information when generating the output.


### Enhanced RAG Pipeline Features

In [33]:
def rag_advanced(query,retriever,llm,top_k=3,min_score=0.2,return_context=False):
    """
    RAG pipeline with extra feature:
    returns answer,sources,confidence_score,context if return_context is True
    """
    results=retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return "No relevant documents found."
    
    
    context="\n\n".join([doc['document'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found."
    
    sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','Unknown')),
        'page':doc['metadata'].get('page_number','Unknown'),
        'score':doc['score'],
        'preview':doc['document'][:200]+'...'
        
    }
        for doc in results
    ]
    
    confidence_score=max([doc['score'] for doc in results])
    
    prompt=f"""
            You are a helpful assistant. Use the following context to answer the question.
            Context: {context}
            Question: {query}
            Answer:
            """
    response=llm.invoke([prompt.format(context=context, query=query)])
    
    output={
        'answer':response.content,
        'sources':sources,
        'confidence_score':confidence_score,
        'context':context
    }
    if return_context:
        output['context']=context
    return output


result=rag_advanced("What is attention mechanism?", rag_retriever, llm, top_k=3, min_score=0.2, return_context=True)

Retrieving top 3 documents for query: 'What is attention mechanism?'
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.47it/s]

Retrieved 2 documents above the score threshold of 0.2.


In [34]:
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence Score:", result['confidence_score'])
print("Context:", result['context'])

Answer: The attention mechanism is a function that maps a query and a set of key-value pairs to an output. It's a way to focus on the most relevant information from a large set of data, by assigning weights to each piece of information based on its relevance to the query.

In more technical terms, the attention mechanism takes in a query vector, a set of key-value pairs, and outputs a weighted sum of the values, where the weights are computed based on the similarity between the query and each key. This allows the model to selectively focus on the most relevant information when making decisions or generating outputs.

The attention mechanism is commonly used in natural language processing (NLP) and machine learning models, such as transformer models, to improve their ability to understand and process long-range dependencies in data.
Sources: [{'source': 'attention.pdf', 'page': 'Unknown', 'score': 0.2714173197746277, 'preview': '3.2 Attention\nAn attention function can be described as m